# Medical Transcription & SOAP Note Generation Pipeline

**Alternative approach demonstrating different engineering choices:**

| Component |  | This notebook |
|-----------|--------------|--------------------|
| ASR | HuggingFace `transformers` pipeline - alternative `faster-whisper` (CTranslate2 backend)  |
| SOAP LLM | Anthropic Claude `tool_use` (primary) |
| Fallback LLM | Ollama (prompt + regex parse) |
| Output extraction | Function calling — schema guaranteed |
| Validation | Semantic similarity + linguistic patterns |

---
### Why these choices?
- **HuggingFace pipeline** - unified model-hub API; swap model in one config line; chunk-and-stride handles long audio natively
- **Anthropic tool_use** - `tool_choice` forces the model to populate a defined schema; eliminates brittle regex JSON extraction; type-safe structured output
- **Semantic validation** - `sentence-transformers` cosine similarity catches *paraphrased* patient-reported content that regex patterns miss


In [1]:
import json
import os
import re
import textwrap
import time
from datetime import datetime
from pathlib import Path

import requests
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer

nltk.download('punkt_tab', quiet=True)
print("\u2713 Imports loaded")


✓ Imports loaded


In [2]:
# ── File paths ────────────────────────────────────────────────────────────
BASE_DIR    = Path(".")
AUDIO_FILE  = BASE_DIR / "sample_dictation.mp3"
OUTPUT_DIR  = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

TRANSCRIPT_FILE = OUTPUT_DIR / "raw_transcript.txt"
SOAP_JSON_FILE  = OUTPUT_DIR / "soap_note.json"
SOAP_MD_FILE    = OUTPUT_DIR / "soap_note.md"
VALIDATION_FILE = OUTPUT_DIR / "validation_report.json"

# ── ASR: HuggingFace transformers Whisper pipeline ─────────────────────────
# Model options (HuggingFace hub IDs):
#   "openai/whisper-base"   — fastest CPU run (~140 MB weights)
#   "openai/whisper-small"  — better accuracy (~460 MB)
#   "openai/whisper-medium" — best medical-term accuracy (~1.5 GB)
WHISPER_HF_MODEL = "openai/whisper-base"
WHISPER_LANGUAGE = "en"

# ── LLM: Anthropic as primary, Ollama as fallback ──────────────────────────
ANTHROPIC_MODEL = "claude-sonnet-4-6"
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "gemma2:2b"
OLLAMA_TIMEOUT  = 300

# ── Semantic validation threshold (cosine similarity) ──────────────────────
SEMANTIC_LEAKAGE_THRESHOLD = 0.80

print(f"Audio     : {AUDIO_FILE}")
print(f"ASR model : {WHISPER_HF_MODEL} (HuggingFace transformers)")
print(f"LLM       : {ANTHROPIC_MODEL} primary | Ollama {OLLAMA_MODEL} fallback")
print(f"Outputs   : {OUTPUT_DIR}")


Audio     : sample_dictation.mp3
ASR model : openai/whisper-base (HuggingFace transformers)
LLM       : claude-sonnet-4-6 primary | Ollama gemma2:2b fallback
Outputs   : outputs


---
## Part A - Audio Transcription (Speech-to-Text)

**Why HuggingFace `transformers` pipeline (vs `faster-whisper`)?**
- Single unified API for all Whisper variants and future models from the hub
- Built-in `chunk_length_s` / `stride_length_s` for long-form audio — no manual batching needed
- `return_timestamps=True` yields chunk-level timestamps out of the box
- PyTorch-native: same framework as the rest of the HuggingFace ecosystem
- Trade-off: slower than CTranslate2 (`faster-whisper`) on CPU; use GPU or reduce model size to compensate


In [3]:
def load_asr_pipeline(model_name: str):
    """Load a HuggingFace Whisper ASR pipeline.

    Args:
        model_name: HuggingFace hub model ID (e.g. 'openai/whisper-base').

    Returns:
        Configured transformers ASR pipeline.
    """
    print(f"Loading HuggingFace Whisper pipeline: '{model_name}' ...")
    print("(First run downloads model weights from HuggingFace hub)")
    t0 = time.time()
    asr = hf_pipeline(
        "automatic-speech-recognition",
        model=model_name,
        chunk_length_s=30,   # split long audio into 30-second chunks
        stride_length_s=5,   # 5-second overlap between chunks for continuity
        device=-1,           # CPU; change to 0 for first GPU
    )
    print(f"\u2713 Pipeline ready in {time.time()-t0:.1f}s")
    return asr


def transcribe_audio(asr, audio_path: Path) -> dict:
    """Transcribe an audio file using the HuggingFace Whisper pipeline.

    Args:
        asr:        Loaded transformers ASR pipeline.
        audio_path: Path to the audio file.

    Returns:
        dict with keys: full_text, segments, language, duration_s.
    """
    print(f"Transcribing: {audio_path.name}")
    t0 = time.time()
    result = asr(
        str(audio_path),
        return_timestamps=True,
        generate_kwargs={"language": WHISPER_LANGUAGE, "task": "transcribe"},
    )
    full_text = result["text"].strip()
    chunks    = result.get("chunks", [])
    segments  = [
        {
            "id":    i,
            "start": round(float(c["timestamp"][0] or 0.0), 2),
            "end":   round(float(c["timestamp"][1] or 0.0), 2),
            "text":  c["text"].strip(),
        }
        for i, c in enumerate(chunks)
    ]
    duration = segments[-1]["end"] if segments else 0.0
    elapsed  = time.time() - t0
    print(f"\u2713 Done in {elapsed:.1f}s | {len(segments)} chunks | {len(full_text.split())} words")
    return {
        "full_text":  full_text,
        "segments":   segments,
        "language":   WHISPER_LANGUAGE,
        "duration_s": round(duration, 2),
    }


asr_pipeline  = load_asr_pipeline(WHISPER_HF_MODEL)
transcription = transcribe_audio(asr_pipeline, AUDIO_FILE)

print("\n" + "=" * 70)
print("RAW TRANSCRIPT (first 800 characters)")
print("=" * 70)
print(transcription["full_text"][:800])
if len(transcription["full_text"]) > 800:
    remaining = len(transcription["full_text"]) - 800
    print(f"\n... [{remaining} more characters] ...")


Loading HuggingFace Whisper pipeline: 'openai/whisper-base' ...
(First run downloads model weights from HuggingFace hub)


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Pipeline ready in 2.9s
Transcribing: sample_dictation.mp3


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

✓ Done in 373.8s | 11 chunks | 1501 words

RAW TRANSCRIPT (first 800 characters)
what was you in? Hi, I've had this pain on the outside of my right elbow now. I first started noticing it several months ago, but recently it's just been more painful. Okay, so you said several months ago. Did anything happen several months ago? Was there any sort of trigger, trauma, anything like that to that area? No, there wasn't any trauma or any triggers that I noticed. I was just feeling it a bit more at the end of work. Yeah, I was just having a feeling of pain a bit more at the end of work. Okay. Does anything make it better or worse, the pain? Yeah, if I really, if I'm just resting the elbow, it makes it better and I've tried things like ibuprofen, which has helped with the pain. I'll do that for helping get through work sometimes if the paint is bad enough. Right, okay. And if yo

... [7125 more characters] ...


In [ ]:
def save_transcript(t: dict, path: Path) -> None:
    """Save the verbatim transcript with metadata header and timed chunks.

    Args:
        t:    Transcription result dict.
        path: Destination .txt file path.
    """
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("MEDICAL DICTATION \u2014 RAW TRANSCRIPT\n")
        fh.write("=" * 60 + "\n")
        fh.write(f"Generated  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        fh.write(f"ASR Model  : {WHISPER_HF_MODEL} (HuggingFace transformers)\n")
        fh.write(f"Language   : {t['language']}\n")
        fh.write(f"Duration   : {t['duration_s']}s\n")
        fh.write(f"Chunks     : {len(t['segments'])}\n")
        fh.write("=" * 60 + "\n\n")
        fh.write(t["full_text"])
        fh.write("\n\n")
        fh.write("=" * 60 + "\n")
        fh.write("TIMED CHUNKS\n")
        fh.write("=" * 60 + "\n")
        for s in t["segments"]:
            fh.write(f'[{s["start"]:>7.2f}s \u2192 {s["end"]:>7.2f}s]  {s["text"]}\n')
    print(f"\u2713 Transcript saved \u2192 {path}")


save_transcript(transcription, TRANSCRIPT_FILE)


---
## Part B - SOAP Note Generation (Text-to-SOAP)

**Why Anthropic Claude with `tool_use` (vs Ollama + prompt engineering)?**

The approach prompts the LLM to output JSON and then uses regex to extract and clean it. This is fragile — models occasionally wrap output in markdown fences, add preamble, or produce malformed JSON.

**`tool_use` / function calling** solves this at the API level:
- `tool_choice: {type: 'tool', name: 'create_soap_note'}` forces the model to invoke the tool — structured output is **guaranteed** by the API
- The tool `input_schema` is a JSON Schema — the model must populate every required field
- Zero regex/parsing code needed in the application layer
- Type-safe: each SOAP section is a validated string field

**Ollama remains as a fallback** for fully-local operation without an API key.

**Prompt engineering strategy (shared by both backends):**
- Explicit S/O/A/P definitions with inclusion/exclusion rules
- 'No duplication across sections' hard rule
- Conversational encounter context (doctor asks, patient answers)


In [ ]:
# Anthropic tool schema — defines the SOAP note structure.
# tool_choice forces the model to return data matching this schema.
# No regex JSON extraction needed — the API guarantees the structure.
SOAP_TOOL = {
    "name": "create_soap_note",
    "description": (
        "Create a structured SOAP note from a physician-patient encounter transcript. "
        "Carefully separate patient-reported information (Subjective) from clinician-observed "
        "findings (Objective). Do NOT repeat the same fact in two sections."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "subjective": {
                "type": "string",
                "description": (
                    "Patient-reported information ONLY: chief complaint, symptom onset/duration, "
                    "pain quality and scale (e.g. 4/10), aggravating/relieving factors, past medical "
                    "and surgical history as stated by patient, current medications, allergies, "
                    "family history, social history (occupation, smoking, alcohol, travel), "
                    "review of systems from patient perspective. "
                    "Use: 'Patient reports', 'Patient states', 'Patient denies'."
                ),
            },
            "objective": {
                "type": "string",
                "description": (
                    "Clinician-performed observations and measurements ONLY: palpation findings, "
                    "range-of-motion and provocative test results performed by the clinician, "
                    "inspection findings, vital signs, lab and imaging results. "
                    "Do NOT include any patient self-reports."
                ),
            },
            "assessment": {
                "type": "string",
                "description": "Clinical diagnosis or impression. One to three sentences.",
            },
            "plan": {
                "type": "string",
                "description": (
                    "Treatment plan: medications with doses and duration, activity modifications, "
                    "referrals, imaging orders, follow-up timeline, patient education."
                ),
            },
        },
        "required": ["subjective", "objective", "assessment", "plan"],
    },
}

SOAP_SYSTEM_PROMPT = (
    "You are a board-certified clinical documentation specialist. "
    "Extract a structured SOAP note from the medical encounter transcript. "
    "The transcript is a real doctor-patient conversation — extract clinical "
    "information from BOTH speakers. "
    "Rules: (1) Patient-reported symptoms -> Subjective ONLY. "
    "(2) Clinician exam findings -> Objective ONLY. "
    "(3) No duplication across sections."
)

print("\u2713 SOAP tool schema and prompt ready")


In [ ]:
def check_anthropic_available() -> bool:
    """Return True if ANTHROPIC_API_KEY is set in the environment."""
    return bool(os.environ.get("ANTHROPIC_API_KEY"))


def generate_soap_anthropic(transcript: str) -> dict:
    """Generate SOAP note via Anthropic Claude using tool_use (function calling).

    JSON extraction is needed. The API guarantees the response matches SOAP schema.

    Args:
        transcript: Raw transcription text.

    Returns:
        Dict with subjective, objective, assessment, plan keys.
    """
    import anthropic
    client = anthropic.Anthropic()
    user_content = f"{SOAP_SYSTEM_PROMPT}\n\nTranscript:\n\"\"\"\n{transcript}\n\"\"\""
    print(f"Generating SOAP via Anthropic ({ANTHROPIC_MODEL}) with tool_use ...")
    t0 = time.time()
    response = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        tools=[SOAP_TOOL],
        tool_choice={"type": "tool", "name": "create_soap_note"},
        messages=[{"role": "user", "content": user_content}],
    )
    elapsed = time.time() - t0
    print(
        f"\u2713 Response in {elapsed:.1f}s | "
        f"tokens in={response.usage.input_tokens} out={response.usage.output_tokens}"
    )
    for block in response.content:
        if block.type == "tool_use" and block.name == "create_soap_note":
            return block.input
    raise RuntimeError("No tool_use block in Anthropic response")


In [ ]:
def check_ollama_available() -> bool:
    """Return True if the Ollama server is reachable."""
    try:
        return requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5).status_code == 200
    except Exception:
        return False


# Ollama still uses prompt engineering + JSON extraction as fallback.
SOAP_OLLAMA_PROMPT = (
    "You are a clinical documentation specialist. "
    "Convert the transcript below into a structured SOAP note.\n\n"
    "DEFINITIONS:\n"
    "  S - SUBJECTIVE: Patient-reported symptoms, history, pain scale, medications.\n"
    "  O - OBJECTIVE: Clinician exam findings, palpation, ROM tests, vitals, imaging.\n"
    "  A - ASSESSMENT: Clinical diagnosis or impression.\n"
    "  P - PLAN: Treatments, prescriptions, referrals, follow-up.\n\n"
    "RULES: No duplication. Output valid JSON only. Each value is a plain string.\n\n"
    "EXAMPLE:\n"
    '{\"subjective\": \"Patient reports sharp right knee pain 7/10 for 2 days. Ibuprofen minimal relief.\",\n'
    ' \"objective\": \"Tenderness on medial joint line. Mild swelling. Limited ROM. X-ray: no fracture.\",\n'
    ' \"assessment\": \"Suspected medial meniscus tear.\",\n'
    ' \"plan\": \"Rest, ice, PT x 4 weeks. MRI if no improvement.\"}\n\n'
    "Transcript:\n\"\"\"\n{transcript}\n\"\"\"\n\n"
    "Output (JSON only):"
)


def extract_json_from_response(raw: str) -> dict:
    """Extract JSON from raw LLM response (handles markdown fences, prose wrapping)."""
    cleaned = re.sub(r"```(?:json)?\s*", "", raw, flags=re.IGNORECASE).replace("```", "").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{[\s\S]+\}", cleaned)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    try:
        return json.loads(re.sub(r",\s*([}\]])", r"\1", cleaned))
    except json.JSONDecodeError:
        raise ValueError(f"Could not parse JSON from response:\n{raw[:400]}")


def generate_soap_ollama(transcript: str) -> dict:
    """Fallback: generate SOAP note using Ollama with prompt engineering."""
    prompt = SOAP_OLLAMA_PROMPT.format(transcript=transcript)
    print(f"Generating SOAP via Ollama ({OLLAMA_MODEL}) [fallback] ...")
    t0 = time.time()
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model":  OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.1, "top_p": 0.9, "num_predict": 1024},
        },
        timeout=OLLAMA_TIMEOUT,
    )
    resp.raise_for_status()
    raw = resp.json()["response"]
    print(f"\u2713 Response in {time.time()-t0:.1f}s")
    return extract_json_from_response(raw)


In [ ]:
anthropic_ok = check_anthropic_available()
ollama_ok    = check_ollama_available()

print(f"Anthropic API key present : {anthropic_ok}")
print(f"Ollama server reachable   : {ollama_ok}")

if anthropic_ok:
    soap_note    = generate_soap_anthropic(transcription["full_text"])
    llm_used     = ANTHROPIC_MODEL
    backend_used = "anthropic_tool_use"
elif ollama_ok:
    print("Anthropic key not set \u2014 falling back to Ollama")
    soap_note    = generate_soap_ollama(transcription["full_text"])
    llm_used     = OLLAMA_MODEL
    backend_used = "ollama_prompt"
else:
    raise RuntimeError(
        "No LLM available. Set ANTHROPIC_API_KEY or start Ollama with `ollama serve`."
    )

# Verify all four sections present
missing = {"subjective", "objective", "assessment", "plan"} - set(soap_note)
if missing:
    raise ValueError(f"SOAP note is missing required sections: {missing}")

SECTION_LABELS = {
    "subjective": "S \u2014 Subjective",
    "objective":  "O \u2014 Objective",
    "assessment": "A \u2014 Assessment",
    "plan":       "P \u2014 Plan",
}

print("\n" + "=" * 70)
print("GENERATED SOAP NOTE")
print("=" * 70)
for key in ["subjective", "objective", "assessment", "plan"]:
    label = SECTION_LABELS[key]
    print(f"\n{label}")
    print("\u2500" * len(label))
    print(textwrap.fill(soap_note.get(key, "[not found]"), width=68))
print("\n" + "=" * 70)


In [ ]:
def save_soap_json(soap: dict, transcript: str, path: Path) -> None:
    """Save SOAP note as JSON with pipeline metadata.

    Args:
        soap:       Dict with SOAP sections.
        transcript: Source transcript (for traceability).
        path:       Destination .json file.
    """
    output = {
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "asr_model":    WHISPER_HF_MODEL,
            "asr_backend":  "HuggingFace transformers pipeline",
            "llm_model":    llm_used,
            "llm_backend":  backend_used,
            "source_audio": AUDIO_FILE.name,
        },
        "transcript": transcript,
        "soap_note":  soap,
    }
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(output, fh, indent=2, ensure_ascii=False)
    print(f"\u2713 SOAP JSON saved \u2192 {path}")


def save_soap_markdown(soap: dict, path: Path) -> None:
    """Save SOAP note as a Markdown document.

    Args:
        soap: Dict with SOAP sections.
        path: Destination .md file.
    """
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = [
        "# SOAP Note",
        f"_Generated: {ts} | ASR: {WHISPER_HF_MODEL} | LLM: {llm_used}_",
        "",
        "---",
        "",
        "## S \u2014 Subjective",
        soap.get("subjective", ""),
        "",
        "## O \u2014 Objective",
        soap.get("objective", ""),
        "",
        "## A \u2014 Assessment",
        soap.get("assessment", ""),
        "",
        "## P \u2014 Plan",
        soap.get("plan", ""),
        "",
    ]
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print(f"\u2713 SOAP Markdown saved \u2192 {path}")


save_soap_json(soap_note, transcription["full_text"], SOAP_JSON_FILE)
save_soap_markdown(soap_note, SOAP_MD_FILE)


---
## Logic Validation: Subjective/Objective Leakage Check

**Two-layer validation :**

**Layer 1 — Linguistic patterns** (fast): Regex scan of the Objective section for explicit patient-report language markers: pronoun + report verbs, pain scales, self-report phrases.

**Layer 2 — Semantic similarity** (deep): Encodes each Objective and Subjective sentence with `sentence-transformers` (`all-MiniLM-L6-v2`) and computes pairwise cosine similarity. An Objective sentence scoring ≥ `SEMANTIC_LEAKAGE_THRESHOLD` against any Subjective sentence is flagged — this catches *paraphrased* patient content that regex misses.

Severity levels:
- **HIGH**: cosine ≥ 0.90 or explicit patient-report verb (`patient reports/states`)
- **MEDIUM**: cosine ≥ 0.80 or pain scale patterns
- **LOW**: lower-confidence overlaps


In [ ]:
_semantic_encoder = None


def get_semantic_encoder() -> SentenceTransformer:
    """Lazy-load the sentence-transformer encoder (cached after first call)."""
    global _semantic_encoder
    if _semantic_encoder is None:
        print("Loading sentence-transformer (all-MiniLM-L6-v2) ...")
        _semantic_encoder = SentenceTransformer("all-MiniLM-L6-v2")
    return _semantic_encoder


def cosine_similarity_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Compute pairwise cosine similarity between two sets of embeddings."""
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-9)
    return a_norm @ b_norm.T


# Linguistic patterns — explicit patient-report markers (Layer 1)
LINGUISTIC_PATTERNS = [
    (r"patient\s+(reports|states|says|feels|complains|describes|denies|mentions)", "HIGH"),
    (r"(he|she|they)\s+(reports|states|says|feels|complains)", "HIGH"),
    (r"\d+\s*(out\s+of|/)\s*10", "HIGH"),
    (r"per\s+patient", "MEDIUM"),
    (r"rates?\s+(the\s+)?pain", "MEDIUM"),
    (r"(no\s+relief|minimal\s+relief|hasn.t\s+helped)", "LOW"),
    (r"(ibuprofen|tylenol|advil|aspirin)\s+(hasn.t|didn.t|not|hasn)", "LOW"),
]


def check_subjective_leakage(soap: dict) -> dict:
    """Detect patient-reported content in the Objective section.

    Two-layer check:
      Layer 1 (Linguistic): Regex patterns for explicit self-report markers.
      Layer 2 (Semantic): Sentence-transformer cosine similarity — catches
        paraphrased content that linguistic patterns miss.

    Args:
        soap: Dict with subjective, objective, assessment, plan.

    Returns:
        Validation report dict with flags and overall PASS/FAIL status.
    """
    obj_text  = soap.get("objective", "")
    subj_text = soap.get("subjective", "")
    flags = []

    # Layer 1: Linguistic pattern check
    for pattern, severity in LINGUISTIC_PATTERNS:
        matches = re.findall(pattern, obj_text, flags=re.IGNORECASE)
        if matches:
            flags.append({
                "layer":    "linguistic",
                "pattern":  pattern,
                "matches":  [m if isinstance(m, str) else str(m) for m in matches],
                "severity": severity,
                "message":  f"Patient-report language pattern in Objective. Pattern: '{pattern}'",
            })

    # Layer 2: Semantic similarity check
    if obj_text.strip() and subj_text.strip():
        obj_sents  = [s for s in sent_tokenize(obj_text)  if len(s.split()) >= 4]
        subj_sents = [s for s in sent_tokenize(subj_text) if len(s.split()) >= 4]
        if obj_sents and subj_sents:
            encoder    = get_semantic_encoder()
            obj_emb    = encoder.encode(obj_sents,  convert_to_numpy=True)
            subj_emb   = encoder.encode(subj_sents, convert_to_numpy=True)
            sim_matrix = cosine_similarity_matrix(obj_emb, subj_emb)
            for i, obj_sent in enumerate(obj_sents):
                max_sim = float(sim_matrix[i].max())
                best_j  = int(sim_matrix[i].argmax())
                if max_sim >= SEMANTIC_LEAKAGE_THRESHOLD:
                    severity = "HIGH" if max_sim >= 0.90 else "MEDIUM"
                    flags.append({
                        "layer":              "semantic",
                        "objective_sentence": obj_sent,
                        "similar_subjective": subj_sents[best_j],
                        "similarity_score":   round(max_sim, 3),
                        "severity":           severity,
                        "message": (
                            f"Objective sentence semantically similar to Subjective content "
                            f"(cosine {max_sim:.2f} \u2265 {SEMANTIC_LEAKAGE_THRESHOLD}). "
                            "Verify this reflects a clinical observation, not patient self-report."
                        ),
                    })

    high   = sum(1 for f in flags if f["severity"] == "HIGH")
    medium = sum(1 for f in flags if f["severity"] == "MEDIUM")
    low    = sum(1 for f in flags if f["severity"] == "LOW")
    passed = (high == 0) and (medium == 0)

    return {
        "validation_timestamp":   datetime.now().isoformat(),
        "validation_method":      "linguistic_patterns + semantic_similarity (all-MiniLM-L6-v2)",
        "semantic_threshold":     SEMANTIC_LEAKAGE_THRESHOLD,
        "overall_status":         "PASS" if passed else "FAIL",
        "high_severity_count":    high,
        "medium_severity_count":  medium,
        "low_severity_count":     low,
        "total_flags":            len(flags),
        "flags":                  flags,
        "objective_section":      obj_text,
    }


In [ ]:
validation_report = check_subjective_leakage(soap_note)

status_icon = "\u2713" if validation_report["overall_status"] == "PASS" else "\u2717"
print("=" * 70)
print("VALIDATION REPORT \u2014 Subjective/Objective Leakage Check")
print("=" * 70)
print(f"Method    : {validation_report['validation_method']}")
print(f"Threshold : cosine ≥ {validation_report['semantic_threshold']} for semantic flags")
print(f"Status    : {status_icon} {validation_report['overall_status']}")
print(f"HIGH      : {validation_report['high_severity_count']}")
print(f"MEDIUM    : {validation_report['medium_severity_count']}")
print(f"LOW       : {validation_report['low_severity_count']}")

if validation_report["flags"]:
    print("\nDetailed flags:")
    for flag in validation_report["flags"]:
        layer = flag["layer"].upper()
        print(f"\n  [{flag['severity']}] [{layer}] {flag['message']}")
        if "similarity_score" in flag:
            print(f"    Obj  : {flag['objective_sentence']}")
            print(f"    Subj : {flag['similar_subjective']}")
            print(f"    Score: {flag['similarity_score']}")
        elif "matches" in flag:
            print(f"    Matches: {flag['matches']}")
else:
    print("\nNo subjective/objective leakage detected. Documentation is clean.")

with open(VALIDATION_FILE, "w", encoding="utf-8") as fh:
    json.dump(validation_report, fh, indent=2, ensure_ascii=False)
print(f"\n\u2713 Validation report saved \u2192 {VALIDATION_FILE}")


In [ ]:
print("=" * 70)
print("PIPELINE COMPLETE \u2014 Output File Inventory")
print("=" * 70)

output_files = [
    (TRANSCRIPT_FILE, "Raw verbatim transcript (plain text + timed chunks)"),
    (SOAP_JSON_FILE,  "Structured SOAP note (JSON with metadata)"),
    (SOAP_MD_FILE,    "Structured SOAP note (Markdown formatted)"),
    (VALIDATION_FILE, "S/O leakage validation report (JSON)"),
]

for path, description in output_files:
    size = path.stat().st_size if path.exists() else 0
    print(f"  {path.name:<30}  {size:>7} bytes  {description}")

print()
print("Pipeline summary :")
print(f"  ASR backend      : HuggingFace transformers pipeline")
print(f"  ASR model        : {WHISPER_HF_MODEL}")
print(f"  SOAP backend     : {backend_used}")
print(f"  SOAP LLM         : {llm_used}")
print(f"  Audio duration   : {transcription['duration_s']}s")
print(f"  Transcript words : {len(transcription['full_text'].split())}")
print(f"  Validation       : {validation_report['overall_status']}")
print(f"  Validation method: {validation_report['validation_method']}")
print("=" * 70)
